In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# OpenQASM 3 e o Qiskit SDK

<details>
<summary><b>Versões dos pacotes</b></summary>

O código desta página foi desenvolvido com os seguintes requisitos.
Recomendamos usar estas versões ou mais recentes.

```
qiskit[all]~=2.3.0
```
</details>

O Qiskit SDK fornece algumas ferramentas para converter entre representações OpenQASM de programas quânticos e a classe QuantumCircuit. Note que essas ferramentas ainda estão em uma fase exploratória de desenvolvimento e continuarão a evoluir conforme o suporte do Qiskit às capacidades de circuitos dinâmicos expressas pelo OpenQASM 3 aumenta.

> **Note:** Esta função ainda está na fase exploratória. Por isso, é provável que a sintaxe e as capacidades evoluam.

## Importar um programa OpenQASM 3 no Qiskit
Você deve instalar o pacote `qiskit_qasm3_import ` para usar esta função. Instale com o seguinte comando.

In [1]:
import qiskit.qasm3

program = """
    OPENQASM 3.0;
    include "stdgates.inc";

    input float[64] a;
    qubit[3] q;
    bit[2] mid;
    bit[3] out;

    let aliased = q[0:1];

    gate my_gate(a) c, t {
      gphase(a / 2);
      ry(a) c;
      cx c, t;
    }
    gate my_phase(a) c {
      ctrl @ inv @ gphase(a) c;
    }

    my_gate(a * 2) aliased[0], q[{1, 2}][0];
    measure q[0] -> mid[0];
    measure q[1] -> mid[1];

    while (mid == "00") {
      reset q[0];
      reset q[1];
      my_gate(a) q[0], q[1];
      my_phase(a - pi/2) q[1];
      mid[0] = measure q[0];
      mid[1] = measure q[1];
    }

    if (mid[0]) {
      let inner_alias = q[{0, 1}];
      reset inner_alias;
    }

    out = measure q;
"""
circuit = qiskit.qasm3.loads(program)
circuit.draw("mpl")

<Image src="../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg" alt="Output of the previous code cell" />

Atualmente, duas funções de alto nível estão disponíveis para importar do OpenQASM 3 para o Qiskit. Essas funções são `load()`, que recebe um nome de arquivo, e `loads()`, que recebe o próprio programa como uma string:

In [2]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dumps

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

dumps(qc)

'OPENQASM 3.0;\ninclude "stdgates.inc";\nbit[2] meas;\nqubit[2] q;\nh q[0];\ncx q[0], q[1];\nbarrier q[0], q[1];\nmeas[0] = measure q[0];\nmeas[1] = measure q[1];\n'

Neste exemplo, definimos um programa quântico usando OpenQASM 3 e usamos `loads()` para convertê-lo diretamente em um QuantumCircuit:

In [3]:
from qiskit import QuantumCircuit
from qiskit.qasm3 import dump

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

f = open("my_file.txt", "w")
dump(qc, f)
f.close()

![Saída da célula de código anterior](../docs/images/guides/interoperate-qiskit-qasm3/extracted-outputs/e805197c-51fb-4216-8d24-ae26633d29bc-0.svg)

## Exportar para OpenQASM 3
Você pode exportar código Qiskit para OpenQASM 3 com `dumps()`, que exporta para uma string, ou `dump()`, que exporta para um arquivo.

### Exemplo com `dumps()`